# 02 — Within-Day Imputation

This notebook mirrors `python/02_imputation.py` with richer documentation and inline visualisations.

## What this step does

Missing 2-hour slots (NaN after artifact removal) are imputed using a **same-time-of-day median** strategy:

- Only gaps of **≤ 3 consecutive NaN slots** are attempted (longer gaps remain NaN — they likely indicate a full day of non-wear).
- The fill value is the median of the **same-hour slots from ± 3 calendar days** for that participant and session.
- At least **3 valid neighbours** must exist; otherwise the slot stays NaN.

Before imputation we construct a **complete time grid**: every (participant, session, day) that appears in the data gets all 12 slot-hour rows, filling any absent slots with NaN.

**Input:** `data/processed/clean/clean_data.parquet`  
**Output:** `data/processed/imputed/imputed_data.parquet`

In [ ]:
import sys
from pathlib import Path

# Allow imports from python/ when running notebook from python/notebooks/
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from config import (
    CLEAN_PARQUET, IMPUTED_PARQUET, LOGS_DIR,
    COL_ID, COL_SESSION, COL_DAY, COL_WKND,
    COL_HR, COL_SLEEP, COL_STEPS,
    ALL_SLOT_HOURS,
    MAX_IMPUTE_GAP, MIN_NEIGHBOR_COUNT, NEIGHBOR_WINDOW_DAYS,
)

LOGS_DIR.mkdir(parents=True, exist_ok=True)
IMPUTED_PARQUET.parent.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    handlers=[
        logging.FileHandler(LOGS_DIR / "02_imputation.log"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger(__name__)

SIGNALS = [COL_HR, COL_SLEEP, COL_STEPS]

print(f"Imputation parameters:")
print(f"  Max gap to impute:     {MAX_IMPUTE_GAP} consecutive slots")
print(f"  Min neighbour count:   {MIN_NEIGHBOR_COUNT}")
print(f"  Neighbour window:      ±{NEIGHBOR_WINDOW_DAYS} days")
print(f"  Signals to impute:     {SIGNALS}")

## Load clean data

We assert that all expected signal columns are present before proceeding.

In [ ]:
log.info(f"Loading {CLEAN_PARQUET}")
df = pd.read_parquet(CLEAN_PARQUET)
log.info(f"  Input rows: {len(df):,}")

for col in SIGNALS:
    assert col in df.columns, f"Missing column: {col}"

df["start_hour"] = df["start_hour"].astype(int)

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nMissing values per signal:")
for sig in SIGNALS:
    n_nan = df[sig].isna().sum()
    print(f"  {sig}: {n_nan:,} NaN ({n_nan / len(df):.1%})")

## Build complete time grid

For every (participant, session, day) combination observed in the clean data, we ensure all 12 slot-hour entries (0, 2, 4, …, 22) exist. Any slots absent from the source data are added as NaN rows. This prevents the imputation logic from skipping slots that were never recorded.

In [ ]:
day_index = df[[COL_ID, COL_SESSION, COL_DAY, COL_WKND]].drop_duplicates()
n_days = len(day_index)
expected_rows = n_days * len(ALL_SLOT_HOURS)

grid = pd.DataFrame(
    [
        {COL_ID: r[COL_ID], COL_SESSION: r[COL_SESSION],
         COL_DAY: r[COL_DAY], COL_WKND: r[COL_WKND], "start_hour": h}
        for _, r in day_index.iterrows()
        for h in ALL_SLOT_HOURS
    ]
)

merged = grid.merge(
    df[[COL_ID, COL_SESSION, COL_DAY, "start_hour"] + SIGNALS],
    on=[COL_ID, COL_SESSION, COL_DAY, "start_hour"],
    how="left",
)

n_added = len(merged) - len(df)
log.info(f"  Complete grid: {len(merged):,} rows ({n_added:,} slots added as NaN)")

print(f"Unique participant-session-days: {n_days:,}")
print(f"Expected rows (12 slots each):   {expected_rows:,}")
print(f"Actual merged rows:              {len(merged):,}")
print(f"Slots added (were missing):      {n_added:,}")

## Imputation

Two helper functions implement the imputation logic:

- `_imputable_mask(nan_mask, max_gap)` — identifies NaN runs that are short enough (≤ `max_gap`) to be imputable. Longer runs of consecutive NaNs are left as-is (e.g., full days of non-wear).
- `_impute_group(group)` — applies same-time-of-day median imputation for one (participant, session) group across all signals simultaneously. Uses a ±`NEIGHBOR_WINDOW_DAYS`-day window of same-hour observations.

In [ ]:
def _imputable_mask(nan_mask: np.ndarray, max_gap: int) -> np.ndarray:
    """Boolean mask of NaN positions that fall within a run of length <= max_gap."""
    n = len(nan_mask)
    out = np.zeros(n, dtype=bool)
    i = 0
    while i < n:
        if nan_mask[i]:
            j = i
            while j < n and nan_mask[j]:
                j += 1
            if j - i <= max_gap:
                out[i:j] = True
            i = j
        else:
            i += 1
    return out


def _impute_group(group: pd.DataFrame) -> pd.DataFrame:
    """Impute all signals for one (participant, session) group."""
    group = group.sort_values([COL_DAY, "start_hour"]).copy()

    sorted_days = sorted(group[COL_DAY].unique())
    n_days = len(sorted_days)
    day_pos = {d: i for i, d in enumerate(sorted_days)}

    for signal in SIGNALS:
        pivot = (
            group.pivot(index=COL_DAY, columns="start_hour", values=signal)
            .reindex(index=sorted_days, columns=ALL_SLOT_HOURS)
        )
        arr = pivot.values.astype(float)   # (n_days x 12)

        for di in range(n_days):
            row = arr[di]
            nan_mask = np.isnan(row)
            if not nan_mask.any():
                continue

            imputable = _imputable_mask(nan_mask, MAX_IMPUTE_GAP)
            for si in np.where(imputable)[0]:
                neighbours = [
                    arr[di + delta, si]
                    for delta in range(-NEIGHBOR_WINDOW_DAYS, NEIGHBOR_WINDOW_DAYS + 1)
                    if delta != 0
                    and 0 <= di + delta < n_days
                    and not np.isnan(arr[di + delta, si])
                ]
                if len(neighbours) >= MIN_NEIGHBOR_COUNT:
                    arr[di, si] = float(np.median(neighbours))

        # Write imputed values back to the group DataFrame
        for di, day in enumerate(sorted_days):
            for si, hour in enumerate(ALL_SLOT_HOURS):
                mask = (group[COL_DAY] == day) & (group["start_hour"] == hour)
                group.loc[mask, signal] = arr[di, si]

    return group

print("Imputation helper functions defined.")

In [ ]:
log.info("Running imputation...")

parts = []
groups = list(merged.groupby([COL_ID, COL_SESSION]))
n_groups = len(groups)

for i, ((pid, sess), grp) in enumerate(groups):
    if i % 500 == 0:
        log.info(f"  {i + 1}/{n_groups}")
    parts.append(_impute_group(grp))

imputed = pd.concat(parts, ignore_index=True)
log.info("Imputation complete.")
print(f"Output shape: {imputed.shape}")

## Results

We report total slots imputed per signal and visualise the before/after HR time series for one example participant to confirm the imputation looks sensible.

In [ ]:
# --- Imputed counts per signal ---
print("Slots imputed per signal:")
for signal in SIGNALS:
    n_imputed = imputed[signal].notna().sum() - merged[signal].notna().sum()
    log.info(f"  {signal}: {n_imputed:,} slots imputed")
    print(f"  {signal}: {n_imputed:,}")

# --- Before/after HR time series for the first participant ---
first_pid = imputed[COL_ID].iloc[0]
first_sess = imputed.loc[imputed[COL_ID] == first_pid, COL_SESSION].iloc[0]

before_sub = merged[(merged[COL_ID] == first_pid) & (merged[COL_SESSION] == first_sess)].sort_values([COL_DAY, "start_hour"])
after_sub  = imputed[(imputed[COL_ID] == first_pid) & (imputed[COL_SESSION] == first_sess)].sort_values([COL_DAY, "start_hour"])

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

x = range(len(before_sub))
axes[0].plot(x, before_sub[COL_HR].values, color="steelblue", linewidth=0.8, alpha=0.85)
axes[0].set_title(f"HR before imputation — participant {first_pid}, session {first_sess}")
axes[0].set_ylabel("HR (bpm)")

axes[1].plot(x, after_sub[COL_HR].values, color="darkorange", linewidth=0.8, alpha=0.85)
axes[1].set_title(f"HR after imputation")
axes[1].set_ylabel("HR (bpm)")
axes[1].set_xlabel("2-hr slot index")

plt.tight_layout()
plt.show()

In [ ]:
imputed.to_parquet(IMPUTED_PARQUET, index=False)
log.info(f"Saved → {IMPUTED_PARQUET}  ({len(imputed):,} rows)")
print(f"Saved imputed data → {IMPUTED_PARQUET}")
print(f"Output shape: {imputed.shape}")